# Retrieval de teses e dissertações que fazem uso de dados públicos

In [1]:
import json
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import re
from sklearn.metrics import confusion_matrix

import xavy.dataframes as xd
import xavy.explore as xe

## Funções

In [2]:
def fillnone(entry, replacement):
    """
    Return `entry` if it is not None. Otherwise, return `replacement`.
    """
    if entry == None:
        return replacement
    else:
        return entry


def extract(usecases, key):
    """
    Given a list of dicts `usecases`, return a list of all properties 
    of the dicts stored under `key`.
    """
    values = [uc[key] for uc in usecases]
    return values


def select(usecases, key, value, exact=True):
    """
    Return a subset of elements in list of dict `usecases` where 
    property stored under `key` has `value`. If `exact` is False,
    select elements whose value (str) has a substring given by
    `value`.
    """
    if exact == True:
        sel = list(filter(lambda uc: uc[key] == value, usecases))
    else:
        sel = list(filter(lambda uc: fillnone(uc[key], '').find(value) != -1, usecases))
    return sel


class translate_dict(dict):
    """
    A dict that returns the key used if no translation was provided for it.
    """
    def __missing__(self,key):
        return key


def read_jsonl(path):
    """
    Read a JSONL file and return a list of dictionaries.

    Parameters
    ----------
    path : str or Path
        Path to the JSONL file.

    Returns
    -------
    list[dict]
        One dictionary per line in the file.
    """
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue  # skip empty lines
            try:
                #records.append(line)
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_num}") from e
    return records


def extract_xml_tag(text: str, tag: str):
    """
    Extract the content of the first XML tag from a string.

    Parameters
    ----------
    text : str
        String containing XML.
    tag : str
        Tag name to extract.

    Returns
    -------
    str | None
        Inner text of the tag, or None if not found.
    """
    try:
        # Wrap text in a root tag to ensure valid XML
        wrapped = f"<root>{text}</root>"
        root = ET.fromstring(wrapped)

        elem = root.find(tag)
        return elem.text if elem is not None else None

    except ET.ParseError:
        return None


def print_teses(teses_df):
    """
    Print title, abstract and keywords for each academic work in DataFrame.
    Expects columns 'titulo', 'resumo' and 'palavras_chave'.
    """
    for i in range(len(teses_df)):
        r = teses_df.iloc[i]
        print(xd.bold(r['titulo']))
        print(r['resumo'])
        print('')
        print('Palavras-chave:', r['palavras_chave'])
        print('')


def binary_encode(class_series):
    """
    Map a str Series of 'Uses public data' and 'Does not use public data'
    into ones and zeros.
    """
    y = pd.Series(np.nan, index=class_series.index)
    y.loc[class_series.str.contains('Uses public data', case=False)] = 1
    y.loc[class_series.str.contains('Does not use public data', case=False)] = 0
    try:
        return y.astype(int)
    except:
        return y


def estimate_precision(df, sample_selected, N_sample, N_nannot, y_true_col='y', y_pred_col='y_pred'):
    """
    Estimate the precision of an ML selection of instances for 
    the population when only a subset was annotated in advance 
    and a random sample of non-annotated instances is passed 
    to the model, that need to be annotated a posteriori.

    Parameters
    ----------
    df : DataFrame
        Annotated and inferred instances. The annotations must 
        figure in column `y_true_col` while the inferred classes
        must be in column `y_pred_col`.
    sample_selected : list-like, shape (2,).
        The number of previously non-annotated instances that 
        were selected by the model and annotated a posteriori 
        as false-positives `sample_selected[0]` and true-
        positives `sample-selected[1]`.
    N_sample : int
        Size of the sample of non-annotated instances that were
        passed to the model.
    N_nannot : int
        Total number (in the population) of instances that were
        not annotated.
    y_true_col : str
        Name of the column in `df` containing the annotated 
        classes of the instaces.
    y_pred_col : str
        Name of the column in `df` containing the classes 
        predicted by the ML model.

    Returns
    -------
    precision : float
        The estimated precision for the population given the
        incomplete measurements.
    precision_dev : float
        Estimated uncertainty on the quantity above.
    """
    
    # Compute the sample fraction over the whole population (works without annotation):
    sample_frac = N_sample / N_nannot

    # Compute confsion matrix for annotated instances:
    cm = confusion_matrix(df[y_true_col], df[y_pred_col])

    # Combine selected annotated instances with selected, non-annotated instances from sample:
    nss = np.array(sample_selected)
    nproj = cm[:, 1] + nss / sample_frac

    # Compute estimated (projected) precision:
    precision = nproj[1] / nproj.sum()
    # Estimate uncertainty:
    precision_dev = np.sqrt(((nproj / nproj.sum()**2)**2 * nss * (1 - nss / N_sample)).sum()) / sample_frac

    return precision, precision_dev

## Listagem de exemplos de organizações que publicam dados públicos

### Carrega dados

In [3]:
# Lê os dados:
with open('../dados/backups/usecases_bkp_2025-12-12.json', 'r') as f:
    data = json.load(f)
usecases_raw = data['data']

In [4]:
# Seleciona casos considerados de uso de dados públicos:
usecases = list(filter(lambda uc: (uc['status_published'] == True) or ((fillnone(uc['comment'], '').find('Ocultado porque') == -1) and (fillnone(uc['comment'], '').find('não atendem aos nossos critérios') == -1)) , usecases_raw))

# Seleciona casos considerados como não consistentes com nosso critérios de coleta:
not_usecases = list(filter(lambda uc: not ((uc['status_published'] == True) or ((fillnone(uc['comment'], '').find('Ocultado porque') == -1) and (fillnone(uc['comment'], '').find('não atendem aos nossos critérios') == -1))) , usecases_raw))

In [5]:
# Cria uma tabela de datasets utilizados:
ds_list = pd.Series(extract(usecases, 'datasets')).explode().dropna().tolist()
ds_df = pd.DataFrame(ds_list)

### Lista de conjuntos de dados

In [6]:
examples_datasets = '; '.join(ds_df['data_name'].sample(10, random_state=179).unique())
examples_datasets

'Votações Nominais e Abertas; Dados Abertos do Governo Federal; Investimentos em Pesquisa - Doenças Tropiciais Negligenciadas e Malária; Dados de crédito de Pessoa Jurídica do Banco Central; Violência doméstica, sexual e/ou outras violências contra as mulheres; Resultados - 2002; Operador aeroportuário - Tarifas Aeroportuárias: Tetos Tarifários e Reajustes Tarifários; Densidade Demográfica de Alagoas; Taxa de alfabetização da população, total e quilombola, de 15 anos ou mais de idade, por sexo, grupos de idade, localização e situação do domicílio; Metadados da Base Siconv'

In [7]:
examples_data_providers = '; '.join(ds_df['data_institution'].sample(20, random_state=78453).unique())
examples_data_providers

'INEP; Ministério dos Direitos Humanos e da Cidadania; Instituto Internacional Arayara; Voos e operações aéreas - Registro de Serviços Aéreos;  Ministério do Desenvolvimento e Assistência Social, Família e Combate à Fome - MDS; Câmara Municipal de Feira de Santana; Banco Central do Brasil; IBGE; MapBiomas; Agência Nacional de Aviação Civil - ANAC; Câmara dos Deputados;  Agência Nacional do Petróleo, Gás Natural e Biocombustíveis - ANP; CAPES; Ministério dos Transportes; INPE; Ministério da Saúde; Instituto Brasileiro de Geografia e Estatística - IBGE'

## Carrega os dados de teses e dissertações

**FAZER**
* Remover duplicados
* Colocar identificador único e passar aos processos do GPT como `custom_id` (talvez usar uri).
* Entender por que algumas teses aparecem mais de uma vez com URIs diferentes e pequenas alterações em algum campo.

In [8]:
# Load data:
teses_df = pd.read_csv('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe.csv')
# Clean abstract:
teses_df['resumo'] = teses_df['resumo'].str.replace('\r\n', ' ').fillna('')
# Export to other formats:
text_data = teses_df['titulo'] + '\n\n' + teses_df['resumo'] + '\n\n' + teses_df['palavras_chave']
teses_records = teses_df[['titulo', 'resumo', 'palavras_chave']].to_dict(orient='records')

## Avaliação de resumos de trabalhos acadêmicos com GPT

In [9]:
from datausage import PublicDataUsageDetector

In [13]:
# Instantiate classifier:
detector = PublicDataUsageDetector('gpt-5-nano', '../keys/openai.txt', 'prompts/open-data_llm-template.md', examples_datasets, examples_data_providers)

In [25]:
ini = 5
n_instances = 995
end = ini + n_instances
batch_file = f'gpt-batch_ufpe-2024_{ini}-{end}.jsonl'
batch_obj = detector.inspect_batch(teses_records[ini:end], save_to=batch_file, id_offset=ini, batch_description='Detection of public data usage on 2024 UFPE academic works (partition).')

In [186]:
batch_info = detector.client.batches.retrieve('batch_694687f6f2188190b4c2dc0caec8fd6d')

In [187]:
batch_info

Batch(id='batch_694687f6f2188190b4c2dc0caec8fd6d', completion_window='24h', created_at=1766230006, endpoint='/v1/chat/completions', input_file_id='file-S1BZSoCo8xpwunSy1JDXUk', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1766231524, error_file_id=None, errors=None, expired_at=None, expires_at=1766316406, failed_at=None, finalizing_at=1766231467, in_progress_at=1766230069, metadata={'description': 'Detection of public data usage on 2024 UFPE academic works (partition).'}, output_file_id='file-TRJc1a8Krxi1DdeTE6rmz5', request_counts=BatchRequestCounts(completed=995, failed=0, total=995), model='gpt-5-nano-2025-08-07', usage={'input_tokens': 1255007, 'output_tokens': 337690, 'total_tokens': 1592697, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens_details': {'reasoning_tokens': 324032}})

In [27]:
batch = detector.client.batches.retrieve(batch_obj.id)
print(batch)

Batch(id='batch_694687f6f2188190b4c2dc0caec8fd6d', completion_window='24h', created_at=1766230006, endpoint='/v1/chat/completions', input_file_id='file-S1BZSoCo8xpwunSy1JDXUk', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1766231524, error_file_id=None, errors=None, expired_at=None, expires_at=1766316406, failed_at=None, finalizing_at=1766231467, in_progress_at=1766230069, metadata={'description': 'Detection of public data usage on 2024 UFPE academic works (partition).'}, output_file_id='file-TRJc1a8Krxi1DdeTE6rmz5', request_counts=BatchRequestCounts(completed=995, failed=0, total=995), model='gpt-5-nano-2025-08-07', usage={'input_tokens': 1255007, 'output_tokens': 337690, 'total_tokens': 1592697, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens_details': {'reasoning_tokens': 324032}})


### Classify a single work on the fly

In [262]:
# Print work's metadata:
i = 4
print('%(titulo)s\n\n%(resumo)s\n\n%(palavras_chave)s' % teses_records[i])

Emergent high-order modularity in the human connectome network

The usage of graphs to model relationships between variables is a common and well- established practice, but, like every model, it has its limitations. In this work, we explore a workaround for one of those limitations: the correct representation of interactions between more than 2 variables. Specifically, we investigate the 3-by-3 interaction between areas of the brain, modeled using 3-uniform hypergraphs. Given that the pairwise network models of the brain exhibit a modular property, i.e., the presence of vertex modules in the graph, our goal here is to determine if the modular property persists in hypergraphs, using modularity as our metric with a threshold set at 0.3. We utilized data from rs-fMRI scans of 1018 young adults, provided by the Human Connectome Project (HCP). For each patient, we obtained 2 rs-fMRI scans. We observed the 3-by-3 interactions between areas of the brain using two information theory metrics: I

In [263]:
# Classify:
detector.inspect(**teses_records[i])

'Uses public data'

## Resultado da avaliação com GPT

In [41]:
# Load GPT results:
gpt_input  = read_jsonl('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-in_0001-1000.jsonl')
gpt_output = read_jsonl('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-out_0001-1000.jsonl')

In [42]:
# Create a GPT classification DataFrame:
gpt_df = pd.DataFrame()
#gpt_df['gpt_titulo'] = pd.Series([extract_xml_tag(g['body']['messages'][0]['content'], 'TITLE') for g in gpt_input])
gpt_df['gpt_titulo'] = pd.Series([re.findall('<TITLE>(.+)</TITLE>', g['body']['messages'][0]['content'])[0] for g in gpt_input])
gpt_df['gpt_id_in']  = pd.Series(extract(gpt_input, 'custom_id'))
gpt_df['gpt_id_out'] = pd.Series(extract(gpt_output, 'custom_id'))
gpt_df['class']      = pd.Series([g['response']['body']['choices'][0]['message']['content'] for g in gpt_output])

# TEMP: remove duplicates:
gpt_df.drop_duplicates(subset='gpt_titulo', inplace=True)

# Consistency check:
assert (gpt_df['gpt_id_in'] == gpt_df['gpt_id_out']).all()
# Integrity test:
assert xd.iskeyQ(gpt_df[['gpt_titulo']])

In [43]:
# Avaliações com títulos duplicados:
gpt_df.loc[gpt_df['gpt_titulo'].duplicated(keep=False)]

,gpt_titulo,gpt_id_in,gpt_id_out,class


In [46]:
# Teses com títulos duplicados:
duplicated_teses_df = teses_df.loc[teses_df['titulo'].duplicated(keep=False)]
xd.print_string_series(duplicated_teses_df.set_index('titulo')['uri'])

Efeitos da criação extensiva de caprinos sobre a comunidade de formigas e seus atributos funcionais morfológicos na Caatinga: https://repositorio.ufpe.br/handle/123456789/56378
Efeitos da criação extensiva de caprinos sobre a comunidade de formigas e seus atributos funcionais morfológicos na Caatinga: https://repositorio.ufpe.br/handle/123456789/58275
O que propôs Bolsonaro? Análise de conteúdo da agenda legislativa e administrativa (2019 - 2022): https://repositorio.ufpe.br/handle/123456789/56345
O que propôs Bolsonaro? Análise de conteúdo da agenda legislativa e administrativa (2019 - 2022): https://repositorio.ufpe.br/handle/123456789/56250
Galectina-9 como biomarcador de severidade em pacientes com COVID-19 leve e grave : implicações imunológicas e clínicas: https://repositorio.ufpe.br/handle/123456789/57770
Galectina-9 como biomarcador de severidade em pacientes com COVID-19 leve e grave : implicações imunológicas e clínicas: https://repositorio.ufpe.br/handle/123456789/57768
O ba

In [47]:
# Junta classificação com dados originais:
classified_teses_df = teses_df.drop_duplicates(subset=['titulo']).copy()
classified_teses_df = classified_teses_df.merge(gpt_df[['gpt_titulo', 'class']], left_on='titulo', right_on='gpt_titulo', how='outer')
# Verifica se alguma avaliação ficou sem dados originais:
missing_class = set(range(0, len(gpt_df))) - set(classified_teses_df.dropna(subset='class').index)
missing_class

set()

In [61]:
# Compute fraction of works that get selected:
n_classified = len(classified_teses_df.dropna(subset='class'))
n_class_1    = (classified_teses_df['class'] == "Uses public data").sum()
n_class_0    = (classified_teses_df['class'] == "Does not use public data").sum()
frac_class_1 = n_class_1 / n_classified
print('Fração de trabalhos classificados como usando dados públicos: {:.1f}%'.format(frac_class_1 * 100))

Fração de trabalhos classificados como usando dados públicos: 15.7%


In [151]:
# Select works that use public data (according to LLM):
public_data_df = classified_teses_df.loc[classified_teses_df['class'] == 'Uses public data']
#public_data_df.to_csv('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-public-data.csv', index=False)

In [62]:
# Select works that do not use public data (according to LLM):
no_public_data_df = classified_teses_df.loc[classified_teses_df['class'] == "Does not use public data"]

In [66]:
# Select a sample of works that do not use public data:
no_public_data_sample_df = no_public_data_df.sample(120, random_state=678)
#no_public_data_sample_df.to_csv('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-no-public-data_s678.csv', index=False)

## Ajuste fino dos prompts

In [10]:
import pandas as pd

import xavy.dataframes as xd

### Carregando dados

In [11]:
# Carrega dados brutos (anotados):
annot_raw_df = pd.read_csv('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-public-data_annot_v02.csv')
# Trabalhos que o LLM não achou que usa dados públicos:
nopublic_df = pd.read_csv('../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_gpt-no-public-data_s678.csv')

In [12]:
n_annot = len(annot_raw_df)
#n_annot = 132 # Verificado manualmente quantos trabalhos foram avaliados.

# Limpeza dos dados:
annot_df = annot_raw_df.copy()
# Seleciona apenas trabalhos anotados:
#annot_df = annot_df.iloc[:n_annot]
print('# trabalhos com anotação:', len(annot_df))
# Remove trabalhos embargados:
#annot_df = annot_df.loc[~annot_df['Obs'].fillna('').str.contains('embargado', case=False)]
#print('# trabalhos não embargados:', len(annot_df))
# Remove trabalhos pulados:
#pulados = annot_df['Obs'].fillna('').str.contains('ver depois', case=False)
#print(xd.bold('Obs. dos trabalhos pulados:'))
#xd.print_string_series(annot_df.loc[pulados, 'Obs'])
#annot_df = annot_df.loc[~pulados]
#print('# trabalhos não pulados:', len(annot_df))
# Add coluna identificando se anotação indicou uso de dados públicos ou não:
#annot_df['annot_public_data'] = (~annot_df['Area'].isnull()).astype(int)
# Corrige manualmente anotações equivocadas:
#reclass_uri = ['https://repositorio.ufpe.br/handle/123456789/56037', 'https://repositorio.ufpe.br/handle/123456789/55857', 'https://repositorio.ufpe.br/handle/123456789/55633']
#annot_df.loc[annot_df['uri'].isin(reclass_uri), 'annot_public_data'] = 0
#print('# trabalhos que usam dados públicos:', annot_df['annot_public_data'].sum())
#print('# trabalhos que não usam:', (1 - annot_df['annot_public_data']).sum())
print('# trabalhos que usam dados públicos:', annot_df['y'].sum())
print('# trabalhos que não usam:', (1 - annot_df['y']).sum())

# trabalhos com anotação: 155
# trabalhos que usam dados públicos: 96
# trabalhos que não usam: 59


### Retesting LLM classification

In [13]:
from datausage import PublicDataUsageDetector

In [14]:
# Prepare testing database (annotated cases selected by 1st trial + rejected by first trial):
test_df = pd.concat([annot_df, nopublic_df], ignore_index=True)
test_df = test_df.drop_duplicates(subset='titulo', keep='first')
test_df.rename({'class': 'class_v01'}, axis=1, inplace=True)
T_works  = 1000
N_annot  = len(annot_df)
N_sample = len(test_df) - N_annot
#annot_records = annot_df[['titulo', 'resumo', 'palavras_chave']].to_dict(orient='records')
#nopublic_records = nopublic_df[['titulo', 'resumo', 'palavras_chave']].to_dict(orient='records')
#test_records = annot_records + nopublic_records
#test_df['y_pred_v01'] = binary_encode(test_df['class_v01'])
test_records = test_df[['titulo', 'resumo', 'palavras_chave']].to_dict(orient='records')
print('# trabalhos selecionados para a 1a filtragem:', T_works)
print('# trabalhos anotados:', N_annot)
print('# trabalhos não anotados (amostra):', N_sample)
print('# trabalhos para avaliar:', len(test_df))

# trabalhos selecionados para a 1a filtragem: 1000
# trabalhos anotados: 155
# trabalhos não anotados (amostra): 110
# trabalhos para avaliar: 265


In [99]:
# Set prompt version:
prompt_version = 'v10'

In [102]:
# Instantiate classifier:
detector = PublicDataUsageDetector('gpt-5', '../keys/openai.txt', f'prompts/open-data_llm-template_{prompt_version}.md', examples_datasets, examples_data_providers)

# Run batch:
if False:
    batch_file = f'../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_prompt-test-{prompt_version}_gpt-in.jsonl'
    batch_description = f'Optimizing prompt for detection of public data usage on 2024 UFPE academic works (prompt {prompt_version}).'
    batch_obj = detector.inspect_batch(test_records, save_to=batch_file, id_prefix=f'ptest-{prompt_version}_', batch_description=batch_description)

Batches can be acessed and monitored here: <https://platform.openai.com/batches>

### Analyzing results

#### ETL results

In [103]:
# Load GPT results:
gpt_input  = read_jsonl(f'../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_prompt-test-{prompt_version}_gpt-in.jsonl')
gpt_output = read_jsonl(f'../dados/dspaces/tccs-dissertacoes-teses-2024-ufpe_prompt-test-{prompt_version}_gpt-out.jsonl')

In [104]:
# Create a GPT classification DataFrame:
gpt_df = pd.DataFrame()
#gpt_df['gpt_titulo'] = pd.Series([extract_xml_tag(g['body']['messages'][0]['content'], 'TITLE') for g in gpt_input])
gpt_df['gpt_titulo'] = pd.Series([re.findall('<TITLE>(.+)</TITLE>', g['body']['messages'][0]['content'])[0] for g in gpt_input])
gpt_df['gpt_id_in']  = pd.Series(extract(gpt_input, 'custom_id'))
gpt_df['gpt_id_out'] = pd.Series(extract(gpt_output, 'custom_id'))
gpt_df['class']      = pd.Series([g['response']['body']['choices'][0]['message']['content'] for g in gpt_output])
gpt_df['y_pred']     = binary_encode(gpt_df['class'])

# TEMP: remove duplicates:
gpt_df.drop_duplicates(subset='gpt_titulo', inplace=True)

# Consistency check:
assert (gpt_df['gpt_id_in'] == gpt_df['gpt_id_out']).all()
# Integrity test:
assert xd.iskeyQ(gpt_df[['gpt_titulo']])

#### Consistency checks

In [105]:
# Avaliações com títulos duplicados:
gpt_df.loc[gpt_df['gpt_titulo'].duplicated(keep=False)]

,gpt_titulo,gpt_id_in,gpt_id_out,class,y_pred


In [106]:
classified_test_df = test_df.merge(gpt_df[['gpt_titulo', 'class', 'y_pred']], left_on='titulo', right_on='gpt_titulo', how='outer')
# Verifica se alguma avaliação ficou sem dados originais:
missing_class = set(range(0, len(gpt_df))) - set(classified_test_df.dropna(subset='class').index)
missing_class

set()

In [107]:
# There should be only two classes: 
gpt_df['class'].value_counts(dropna=False)

class
Does not use public data    163
Uses public data            102
Name: count, dtype: int64

#### Analyze results

Let's see if any works previously not classified as using public datasets were now classified this way:

In [108]:
# Select such works:
newly_sel_df = classified_test_df.loc[classified_test_df['y'].isnull() & (classified_test_df['y_pred'] == 1)]
# Output:
print_teses(newly_sel_df)

Aspectos mecânicos sobre resina impressa 3D para confecção de restaurações  provisórias: uma revisão integrativa da literatura
Revisar a literatura de forma integrativa através dos aspectos mecânicos sobre resina  impressa 3D para confecção de restaurações provisórias. Metodologia: A revisão de literatura  foi baseada na estratégia PICO. Houve uma buscas de artigos em português e inglês entre os  meses de maio a outubro de 2023 na Biblioteca Virtual em Saúde (BVS); Pubmed, Cochrane  Library e Science Direct. Os termos-chave da estratégia de busca foram baseados em  descritores, sinônimos, termos presentes em título e resumo. Também foi realizada uma busca  manual através das referências apresentadas pelos artigos incluídos na revisão. Os critérios de  inclusão foram estudos com artigos de pesquisa in Vivo, in Vitro ou in Silico, que promoveu a  comparação do aspecto mecânico entre as resinas impressas 3D e resinas convencionais para  confecção de restaurações provisórias em prótese fix

Baseline precision (prompt used for selecting works that were annotated):

In [109]:
baseline_precision = annot_df['y'].mean()
print('Baseline precision: {:.4f}'.format(baseline_precision))

Baseline precision: 0.6194


In [111]:
# Compute new precision over annotated instances:
annot_test_df = classified_test_df.loc[~classified_test_df['y'].isnull()]
blind_test_df = classified_test_df.loc[classified_test_df['y'].isnull()]
precision = annot_test_df.loc[annot_test_df['y_pred'].astype(int) == 1, 'y'].mean()
print('Precision on the annotated set: {:.4f}'.format(precision))

prec, prec_dev = estimate_precision(annot_test_df, [0,2], 110, 1000)
print('Precision on the population = {:.4f} +/- {:.4f}'.format(prec, prec_dev))

Precision on the annotated set: 0.8200
Precision on the population = 0.8477 +/- 0.0914


In [112]:
n11 = confusion_matrix(annot_test_df['y'], annot_test_df['y_pred'])[1,1]
print('# casos corretamente selecionados: {:d}'.format(n11))

# casos corretamente selecionados: 82


In [113]:
# Loss of previously selected cases:
(annot_test_df.loc[annot_test_df['y'].astype(int) == 1, 'y_pred'].astype(int) == 0).mean()

np.float64(0.14583333333333334)

Vamos verificar o que os falsos negativos têm que fizeram-os serem perdidos (talvez não haja uso de dados públicos):

In [88]:
# Salvando trabalhos removidos pelo modelo v04:
#false_negatives[['titulo', 'resumo', 'uri']].to_csv('~/temp/cordata_tccs-ufpe-removidos-por-gpt5.csv', index=False)

In [89]:
false_negatives = annot_test_df.loc[(annot_test_df['y_pred'].astype(int) == 0) & (annot_test_df['y'].astype(int) == 1)]
print_teses(false_negatives)

Evolução dos padrões de coloração da pelagem em marsupiais didelfídeos (Didelphimorphia: Didelphidae)
A ordem Didelphimorphia possui ampla diversidade de modos de locomoção, tamanho corporal e coloração da pelagem. Esta carece de estudos e explicações mais detalhadas sobre sua história evolutiva e função adaptativa. Os poucos estudos com este enfoque associam tonalidades de pelagem em Didelphidae a aspectos ambientais e filogenéticos e consideram os padrões de coloração mais uniformes como estados de caráter plesiomórficos em comparação aos padrões mais complexos. Utilizamos dados da literatura, uma filogenia recente da família e métodos filogenéticos comparativos (PCM) para reconstruir os estados ancestrais dos padrões de coloração do dorso, ventre, cauda, máscara ocular, listra médio-rostral e manchas supraoculares em Didelphidae a partir dos padrões das espécies viventes; testar a correlação entre os padrões de coloração e dados ecológicos (tipo de habitat, tipo de locomoção e padrã

# Trash

## Avaliação de resumos de trabalhos acadêmicos com BERT

### Teste de modelos de IA

In [6]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

#### 1. bertimbau-large-InferBr-NLI

In [96]:
# 1. Load tokenizer and model from HuggingFace
tokenizer = AutoTokenizer.from_pretrained("felipesfpaula/bertimbau-large-InferBr-NLI")
model = AutoModelForSequenceClassification.from_pretrained("felipesfpaula/bertimbau-large-InferBr-NLI")

# 2. Encode a premise–hypothesis pair
premise = text_data.iloc[:2].tolist()
hypothesis = ['Este trabalho trata de violência urbana'] * 2
encoded = tokenizer(premise, hypothesis, return_tensors="pt", max_length=512, truncation=True, padding="max_length")

# 3. Run inference
with torch.no_grad():
    outputs = model(**encoded)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    answer = dict(zip(['Contradiction', 'Entailment', 'Neutral'], list(probs.T.tolist())))

answer

{'Contradiction': [6.719219527440146e-05, 0.0008191487286239862],
 'Entailment': [9.625975508242846e-05, 0.02135404385626316],
 'Neutral': [0.9998365640640259, 0.9778268337249756]}

#### 2. xlm_roberta_base_assin_fine_tuned 

In [42]:
from transformers import XLMRobertaTokenizer

In [43]:
model_path = "giotvr/portuguese-nli-3-labels"
premise = "As mudanças climáticas são uma ameaça séria para a biodiversidade do planeta."
hypothesis ="A biodiversidade do planeta é seriamente ameaçada pelas mudanças climáticas."
tokenizer = XLMRobertaTokenizer.from_pretrained(model_path, use_auth_token=True)
input_pair = tokenizer(premise, hypothesis, return_tensors="pt",padding=True, truncation=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, use_auth_token=True)

with torch.no_grad():
    logits = model(**input_pair).logits
probs = torch.nn.functional.softmax(logits, dim=-1)
probs, sorted_indices = torch.sort(probs, descending=True)
for i, score in enumerate(probs[0]):
    print(f"Class {sorted_indices[0][i]}: {score.item():.4f}")


/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2077: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Class 2: 0.8273
Class 1: 0.1502
Class 0: 0.0226


In [60]:
model_path = "giotvr/portuguese-nli-3-labels"
premise = "As mudanças climáticas são uma ameaça séria para a biodiversidade do planeta."
hypothesis ="A biodiversidade do planeta é seriamente ameaçada pelas mudanças climáticas."
tokenizer = AutoTokenizer.from_pretrained(model_path, use_auth_token=True)
input_pair = tokenizer(premise, hypothesis, return_tensors="pt",padding=True, truncation=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, use_auth_token=True)

with torch.no_grad():
    logits = model(**input_pair).logits
probs = torch.nn.functional.softmax(logits, dim=-1)
probs, sorted_indices = torch.sort(probs, descending=True)
for i, score in enumerate(probs[0]):
    print(f"Class {sorted_indices[0][i]}: {score.item():.4f}")


/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/models/auto/tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Class 2: 0.8273
Class 1: 0.1502
Class 0: 0.0226


In [61]:
input_pair

{'input_ids': tensor([[     0,   1301, 119666,    501,   3141,  28447,      7,   3854,    788,
         165725,   6352,    399,    121,     10,      6, 179174,  16018,     54,
          33910,      5,      2,      2,     62,      6, 179174,  16018,     54,
          33910,    393,  17594,   1183, 165725,     85,  35335, 119666,    501,
           3141,  28447,      7,      5,      2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

#### 3. xlm-roberta-large-xnli

In [3]:
from transformers import pipeline

In [4]:
classifier = pipeline("zero-shot-classification",
                      model="joeddav/xlm-roberta-large-xnli")

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [46]:
# pose sequence as a NLI premise and label as a hypothesis
nli_model = AutoModelForSequenceClassification.from_pretrained('joeddav/xlm-roberta-large-xnli')
tokenizer = AutoTokenizer.from_pretrained('joeddav/xlm-roberta-large-xnli')

premise = text_data.iloc[1]
hypothesis = 'Um dos tópicos desse documento é política.'

# run through model pre-trained on MNLI
x = tokenizer.encode(premise, hypothesis, return_tensors='pt',
                     truncation_strategy='only_first')
logits = nli_model(x.to('cpu'))[0]

# we throw away "neutral" (dim 1) and take the probability of
# "entailment" (2) as the probability of the label being true 
entail_contradiction_logits = logits[:,[0,2]]
probs = entail_contradiction_logits.softmax(dim=1)
prob_label_is_true = probs[:,1]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/home/hxavier/ceweb/projetos/cordata/analises/env/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2869: FutureWarning: The `truncation_strategy` argument is deprecated and will be removed in a future version, use `truncation=True` to truncate examples to a max length. You c

In [16]:
logits[:, [0,2]].softmax(dim=1)

tensor([[0.0032, 0.9968]], grad_fn=<SoftmaxBackward0>)

In [14]:
logits[:, [0,2]]

tensor([[-3.4848,  0.5389,  2.2455]], grad_fn=<AddmmBackward0>)

In [41]:
# load model pretrained on MNLI
# pose sequence as a NLI premise and label (politics) as a hypothesis
premise = 'Who are you voting for president of the USA in 2020?'
hypothesis = 'This text is about sex.'
#antithesis = 'This text is not about sex.'

# run through model pre-trained on MNLI
input_ids = tokenizer.encode(premise, hypothesis, return_tensors='pt')
logits = nli_model(input_ids)[0]

# we throw away "neutral" (dim 1) and take the probability of
# "entailment" (2) as the probability of the label being true 
entail_contradiction_logits = logits[:,[0,2]]
probs = entail_contradiction_logits.softmax(dim=1)
true_prob = probs[:,1].item() * 100
#print(f'Probability that the label is true: {true_prob:0.2f}%')

nli_model(input_ids)[0].softmax(dim=1)

tensor([[0.9820, 0.0039, 0.0141]], grad_fn=<SoftmaxBackward0>)

In [59]:
input_ids

tensor([[    0, 40469,   621,   398, 20875,   214,   100, 13918,   111,    70,
          4602,    23, 11075,    32,     2,     2,  3293,  7986,    83,   959,
          1672,  1100,     5,     2]])

In [109]:
class NuclearNLI:

    def __init__(self, model_path):
        self.model_path = model_path
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        if model_path == 'felipesfpaula/bertimbau-large-InferBr-NLI':
            self.max_length = 128
            self.labels = ['Contradiction', 'Entailment', 'Neutral']
        else:
            self.max_length = 512
            self.labels = ['Contradiction', 'Neutral', 'Entailment']

    def __call__(self, premise, hypothesis):
        input_pair = self.tokenizer(premise, hypothesis, return_tensors='pt', padding='max_length', truncation=True, max_length=self.max_length)
        with torch.no_grad():
            output = self.model(**input_pair)
            logits = output.logits
            probs  = logits.softmax(dim=1)
        return probs

   

In [124]:
#m = NuclearNLI('giotvr/portuguese-nli-3-labels', ['Contradiction', 'Neutral', 'Entailment'])
#m = NuclearNLI('joeddav/xlm-roberta-large-xnli', ['Contradiction', 'Neutral', 'Entailment'])
m = NuclearNLI('felipesfpaula/bertimbau-large-InferBr-NLI')

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("hapaxlegomenon/InferBR")
ds_train = ds['train']
ds_test  = ds['test']
ds_val   = ds['validation']

df_test = ds_test.to_pandas()
df_val  = ds_val.to_pandas()

In [136]:
pred_test = m(df_test['premise'].tolist(), df_test['hypothesis'].tolist()).argmax(dim=1)

In [150]:
pred_val = m(df_val['premise'].tolist(), df_val['hypothesis'].tolist()).argmax(dim=1)

In [143]:
pred_test

tensor([0, 2, 1,  ..., 2, 0, 1])

In [153]:
from sklearn.metrics import f1_score, precision_score

In [146]:
f1_score(df_test['label'], pred_test, average='macro')

0.9395691597747202

In [151]:
f1_score(df_val['label'], pred_val, average='macro')

0.9383888316788561

In [154]:
precision_score(df_test['label'], pred_test, average='macro')

0.9395668031793686

In [155]:
df_test['label'].value_counts()

label
0    573
1    568
2    564
Name: count, dtype: int64